# Объяснятель: выравнивание Qwen3-4B-Instruct-2507

Полностью воспроизводимый Colab-пайплайн: SFT → style-DPO, отдельная Reward Model, затем независимые quality-DPO и SimPO.

## 0. Цель и инварианты

- Все random seeds равны `42`.
- Обучение выполняется через QLoRA 4-bit на T4.
- `public_test_*` не участвуют в обучении.
- Style-генерация выполняется с `do_sample=False`.
- Все пять метрик вычисляются внутри этого ноутбука.
- DPO и SimPO по качеству ветвятся от одного style-DPO чекпоинта.

## 1. Настройка окружения

In [ ]:
%pip install -q \
  transformers==4.57.3 trl==0.24.0 peft==0.17.1 \
  accelerate==1.11.0 bitsandbytes==0.48.2 datasets==4.3.0 \
  scikit-learn==1.7.2 safetensors==0.6.2 jinja2==3.1.6

In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import bitsandbytes as bnb
import datasets
import numpy as np
import peft
import sklearn
import torch
import transformers
import trl
from transformers import set_seed

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), "Для полного прогона требуется GPU T4"
print("Python:", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
subprocess.run(["nvidia-smi"], check=True)

## 2. Данные и проверки

Загрузка официального архива, проверка SHA-256 и схем JSONL.

In [ ]:
ASSET_URL = "https://edu.tbank.ru/files/c1005bf0-8695-451a-9616-87c8687dce27"
RUNTIME_ROOT = Path("/content/alignment_explainer")
ARCHIVE_PATH = RUNTIME_ROOT / "ml-olympiad-red-task.zip"
EXTRACT_DIR = RUNTIME_ROOT / "source"
CHECKPOINT_DIR = RUNTIME_ROOT / "checkpoints"

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    print("Downloading organizer assets...")
    urllib.request.urlretrieve(ASSET_URL, ARCHIVE_PATH)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

kid_matches = list(EXTRACT_DIR.rglob("kid_adult.jsonl"))
assert len(kid_matches) == 1, f"Expected one kid_adult.jsonl, found {kid_matches}"
PROJECT_ROOT = kid_matches[0].parent.parent
DATA_DIR = PROJECT_ROOT / "data"
METRICS_DIR = PROJECT_ROOT / "metrics"
print("Assets root:", PROJECT_ROOT)

In [ ]:
EXPECTED_SHA256 = {
    "data/good_bad.jsonl": "bf50f3af0127df71d891c5a65eb75220104157f3e27b613aacbae1761c08998b",
    "data/kid_adult.jsonl": "52bacff1c6d5d50ca3dd473f8d754cf1dfcce7e02ecf162cda2c18719a138748",
    "data/public_test_quality.jsonl": "bc8b21bf04c88e99d420569c61f46309f71d04601159b80ea258760e8d871780",
    "data/public_test_style.jsonl": "d0f5fb848245b18e97b97fe5158c602f3f2b49b8ec6588f93a0f0e9f10c58efe",
    "metrics/style_clf.pkl": "b5cf7b53417033de19b9c44a43402bce0e6eeece44b1abac2cf596785b60888d",
    "metrics/gold_rm/adapter_config.json": "e323f8652b0d1b571163c21db0175ac32bee06bc48022b82cd1f7d7c1e94b8fd",
    "metrics/gold_rm/adapter_model.safetensors": "fbfa95e616bc88f6f17da81067390053c938e1e36448cf65b41499603ce2d704",
    "metrics/gold_rm/value_head.safetensors": "4feddd918c31985e644c143c21a49b6739731f6e2eb78e858801109196505f08",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for relative_path, expected_hash in EXPECTED_SHA256.items():
    asset_path = PROJECT_ROOT / relative_path
    assert asset_path.exists(), f"Missing asset: {relative_path}"
    actual_hash = sha256_file(asset_path)
    assert actual_hash == expected_hash, (relative_path, actual_hash, expected_hash)
print(f"Verified {len(EXPECTED_SHA256)} organizer assets by SHA-256")

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

kid_adult_rows = read_jsonl(DATA_DIR / "kid_adult.jsonl")
good_bad_rows = read_jsonl(DATA_DIR / "good_bad.jsonl")
public_style_rows = read_jsonl(DATA_DIR / "public_test_style.jsonl")
public_quality_rows = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

EXPECTED_SCHEMAS = {
    "kid_adult": (kid_adult_rows, 1489, {"prompt", "kid", "adult"}),
    "good_bad": (good_bad_rows, 2226, {"instruction", "chosen", "rejected"}),
    "public_style": (public_style_rows, 50, {"prompt", "kid", "adult", "fact"}),
    "public_quality": (public_quality_rows, 50, {"prompt", "chosen", "rejected"}),
}
for name, (rows, expected_count, expected_keys) in EXPECTED_SCHEMAS.items():
    assert len(rows) == expected_count, (name, len(rows), expected_count)
    assert all(set(row) == expected_keys for row in rows), f"Unexpected schema in {name}"
    assert all(isinstance(value, str) and value.strip() for row in rows for value in row.values())

assert not ({row["prompt"] for row in kid_adult_rows} & {row["prompt"] for row in public_style_rows})
quality_group_sizes = Counter(row["instruction"] for row in good_bad_rows)
print({name: len(rows) for name, (rows, _, _) in EXPECTED_SCHEMAS.items()})
print("Unique quality instructions:", len(quality_group_sizes))
print("Pairs per quality instruction:", Counter(quality_group_sizes.values()))
print("Public datasets are evaluation-only and are not included in any training variable.")

## 3. Формальные измерители

Style probability, pairwise reward accuracy и length-normalized implicit-preference accuracy.

In [ ]:
import pickle
from scipy.sparse import hstack

with (METRICS_DIR / "style_clf.pkl").open("rb") as file:
    style_bundle = pickle.load(file)

style_classifier = style_bundle["clf"]
style_vectorizers = style_bundle["vecs"]
assert list(style_classifier.classes_) == [0, 1]
assert len(style_vectorizers) == 2

def simple_probabilities(texts: list[str]) -> np.ndarray:
    """Return P(simple) for response text only; prompts must not be included."""
    features = hstack(
        [vectorizer.transform(texts) for vectorizer in style_vectorizers],
        format="csr",
    )
    simple_index = list(style_classifier.classes_).index(1)
    return style_classifier.predict_proba(features)[:, simple_index]

sanity_rows = kid_adult_rows[:256]
kid_sanity = simple_probabilities([row["kid"] for row in sanity_rows]).mean()
adult_sanity = simple_probabilities([row["adult"] for row in sanity_rows]).mean()
assert kid_sanity > adult_sanity + 0.50, (kid_sanity, adult_sanity)
print(f"Style scorer sanity: kid={kid_sanity:.4f}, adult={adult_sanity:.4f}")

In [ ]:
def interval_letter(value: float, metric_kind: str) -> str:
    """Map metrics to the task's half-open intervals."""
    assert np.isfinite(value) and 0.0 <= value <= 1.0, value
    if metric_kind == "style":
        bounds = ((0.4, "А"), (0.7, "Б"), (0.9, "В"), (float("inf"), "Г"))
    elif metric_kind == "quality":
        bounds = ((0.6, "А"), (0.75, "Б"), (0.9, "В"), (float("inf"), "Г"))
    else:
        raise ValueError(f"Unknown metric kind: {metric_kind}")
    return next(letter for upper_bound, letter in bounds if value < upper_bound)

def strict_pairwise_accuracy(chosen_scores, rejected_scores) -> tuple[float, int, int]:
    chosen = np.asarray(chosen_scores, dtype=np.float64)
    rejected = np.asarray(rejected_scores, dtype=np.float64)
    assert chosen.shape == rejected.shape and chosen.ndim == 1 and chosen.size > 0
    correct = chosen > rejected  # ties are errors
    return float(correct.mean()), int(correct.sum()), int(correct.size)

METRICS = {}

def record_metric(task: str, value: float, metric_kind: str, **details) -> dict:
    result = {"value": float(value), "interval": interval_letter(value, metric_kind), **details}
    METRICS[task] = result
    print(f"{task}: {value:.6f} -> {result['interval']}")
    return result

assert [interval_letter(x, "quality") for x in (0.59, 0.60, 0.75, 0.90)] == ["А", "Б", "В", "Г"]

In [ ]:
import torch.nn.functional as F

def encode_prompt_completion(tokenizer, prompt: str, completion: str) -> tuple[list[int], list[bool]]:
    """Tokenize an exact chat and mark assistant content, excluding prompt and <|im_end|>."""
    prompt_messages = [{"role": "user", "content": prompt}]
    full_messages = prompt_messages + [{"role": "assistant", "content": completion}]
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    )
    prompt_ids = list(prompt_ids)
    full_ids = list(full_ids)
    assert full_ids[: len(prompt_ids)] == prompt_ids, "Chat prompt is not a prefix of full conversation"

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    assert im_end_id != tokenizer.unk_token_id
    answer_start = len(prompt_ids)
    answer_end = full_ids.index(im_end_id, answer_start)
    assert answer_end > answer_start, "Empty assistant completion"
    completion_mask = [False] * len(full_ids)
    completion_mask[answer_start:answer_end] = [True] * (answer_end - answer_start)
    return full_ids, completion_mask

def mean_completion_logprobs(
    model, tokenizer, prompts: list[str], completions: list[str], *, batch_size: int = 2, max_length: int = 512
) -> np.ndarray:
    """Mean conditional logprob over assistant-content tokens only."""
    assert len(prompts) == len(completions) and prompts
    encoded = [encode_prompt_completion(tokenizer, p, c) for p, c in zip(prompts, completions)]
    too_long = [len(ids) for ids, _ in encoded if len(ids) > max_length]
    assert not too_long, f"Increase max_length; observed lengths: {too_long}"

    pad_token_id = tokenizer.pad_token_id
    assert pad_token_id is not None
    device = next(model.parameters()).device
    scores = []
    was_training = model.training
    model.eval()
    try:
        for start in range(0, len(encoded), batch_size):
            batch = encoded[start : start + batch_size]
            width = max(len(ids) for ids, _ in batch)
            input_ids = []
            attention_masks = []
            completion_masks = []
            for ids, mask in batch:
                padding = width - len(ids)
                input_ids.append(ids + [pad_token_id] * padding)
                attention_masks.append([1] * len(ids) + [0] * padding)
                completion_masks.append(mask + [False] * padding)

            input_tensor = torch.tensor(input_ids, dtype=torch.long, device=device)
            attention_tensor = torch.tensor(attention_masks, dtype=torch.long, device=device)
            answer_mask = torch.tensor(completion_masks, dtype=torch.bool, device=device)[:, 1:]
            with torch.inference_mode():
                logits = model(
                    input_ids=input_tensor, attention_mask=attention_tensor, use_cache=False
                ).logits[:, :-1, :]
                labels = input_tensor[:, 1:]
                token_nll = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)), labels.reshape(-1), reduction="none"
                ).view_as(labels)
                token_logprobs = -token_nll
                token_counts = answer_mask.sum(dim=1)
                assert torch.all(token_counts > 0)
                batch_scores = (token_logprobs * answer_mask).sum(dim=1) / token_counts
                scores.extend(batch_scores.float().cpu().tolist())
            del input_tensor, attention_tensor, answer_mask, logits, labels, token_nll
    finally:
        if was_training:
            model.train()
    return np.asarray(scores, dtype=np.float64)

## 4. Задача 1 — SFT

## 5. Задача 2 — DPO по стилю

## 6. Задача 3 — Reward Model

## 7. Задача 4 — DPO по качеству

## 8. Задача 5 — SimPO

## 9. Итоговые метрики

Финальная таблица должна печатать точные значения и буквы интервалов для всех пяти задач.